# <h1 align="center">LegalMind — EDA</h1>

**Imports**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from datasets import load_dataset
from collections import Counter

**Loading dataset**

In [ ]:
ds = load_dataset("coastalcph/lex_glue", "ecthr_a")

**Dataset Structure**

In [ ]:
print("Structure of ds:", ds)
print(ds['train'].features)

- The dataset has 9,000 training cases, 1,000 test cases, and 1,000 validation cases, where each case is a list of text paragraphs labeled with which of 10 possible ECHR articles (if any) were violated.

**One record check**

In [ ]:
print(ds['train'][0])

- This first case is quite long (93 paragraphs) and involves a family/child-welfare dispute in Finland, labeled with a single violation: **label `4`**, which corresponds to **Article 8** (right to private and family life) — makes sense given the facts are all about custody, access restrictions, and public care of children.

**Text Length Analysis**

In [ ]:
lengths = [len(case) for case in ds['train']['text']]
print(f"\n=== CASE LENGTH ===")
print(f"Mean: {np.mean(lengths):.1f} paragraphs")
print(f"Min: {min(lengths)}, Max: {max(lengths)}")

- The average case is about 24 paragraphs, but the range is huge (from 1 to 558) - dataset has high variance in case length.

**Text Length Analysis - Visualization**

In [ ]:
lengths = [len(case) for case in ds['train']['text']]

plt.figure(figsize=(10,5))
plt.hist(lengths, bins=50, color='steelblue', edgecolor='black')
plt.title('Case Length Distribution (Number of Paragraphs)')
plt.xlabel('Number of Paragraphs')
plt.ylabel('Number of Cases')
plt.axvline(np.mean(lengths), color='red', linestyle='--', label=f'Mean: {np.mean(lengths):.1f}')
plt.legend()
plt.tight_layout()
plt.show()

**Text Length Analysis — Characters/Tokens**

In [ ]:
char_lengths = [len(" ".join(case)) for case in ds['train']['text']]

print(f"Mean: {np.mean(char_lengths):.0f} chars")
print(f"Min: {min(char_lengths)}, Max: {max(char_lengths)}")
print(f"Approx tokens (chars/4): mean={np.mean(char_lengths)/4:.0f}, max={max(char_lengths)/4:.0f}")

- Mean case: ~2,450 tokens; max: ~52,500 tokens (21x mean) — chunking should be based on character/token count, because max will be too big for one prompt

**Article Distribution**

In [ ]:
all_labels = [l for case in ds['train']['labels'] for l in case]
counter = Counter(all_labels)
articles = ['2','3','5','6','8','9','10','11','14','P1-1']
print("=== ARTICLE DISTRIBUTION ===")
for idx, count in sorted(counter.items()):
    print(f"Article {articles[idx]}: {count} cases")

- Article 6 dominates (4,704 cases), Article 9 is almost absent (41 cases) — a ~115x imbalance. Aggregate accuracy alone will be misleading; per-article metrics are needed to check if the Judge will genuinely reason or just default to Article 6.

**Article Distribution - Visualization**

In [ ]:
articles = ['2','3','5','6','8','9','10','11','14','P1-1']

all_labels = [l for case in ds['train']['labels'] for l in case]
counter = Counter(all_labels)
counts = [counter[i] for i in range(len(articles))]

plt.figure(figsize=(10,5))
plt.bar(articles, counts, color='steelblue')
plt.title('Distribution of ECHR Articles in the Dataset')
plt.xlabel('Article')
plt.ylabel('Number of Cases')
plt.tight_layout()
plt.show()

**Multi-label Analysis**

In [ ]:
multi = [len(l) for l in ds['train']['labels']]
print(f"\n=== NUMBER OF VIOLATIONS PER CASE ===")
print(f"Mean: {np.mean(multi):.1f}")
print(f"Only 1 violation: {sum(1 for x in multi if x==1)}")
print(f"2+ violations: {sum(1 for x in multi if x>1)}")

- Most cases have 1 violation (5,924), but ~24% (2,162) have 2+.
- Multi-label handling is needed — the Judge must be able to output multiple articles, not just the single best guess.

**Violation vs No-Violation Baseline**

In [ ]:
no_violation = sum(1 for l in ds['train']['labels'] if len(l) == 0)
has_violation = sum(1 for l in ds['train']['labels'] if len(l) > 0)

print(f"No violation: {no_violation} ({no_violation/len(ds['train'])*100:.1f}%)")
print(f"Has violation: {has_violation} ({has_violation/len(ds['train'])*100:.1f}%)")

- Dataset is heavily skewed toward "violation found" (89.8%). Evaluation must check the Judge's no-violation cases specifically (10.2% of data) — a Judge with "violation" would score deceptively well on aggregate accuracy.

**Co-occurrence Analysis**

In [ ]:
pairs = Counter()
for labels in ds['train']['labels']:
    if len(labels) > 1:
        for i in range(len(labels)):
            for j in range(i+1, len(labels)):
                pairs[(labels[i], labels[j])] += 1
print("=== TOP CO-OCCURRENCE ===")
articles = ['2','3','5','6','8','9','10','11','14','P1-1']
for (a,b), count in pairs.most_common(5):
    print(f"Art.{articles[a]} + Art.{articles[b]}: {count}x")


- Article 6 + P1-1 co-occur far more than any other pair (958x) — likely because fair-trial issues often stem from property-related legal disputes. 
- For evaluation: If the Judge flags one, it should often consider the other too 

**Co-occurance Visualization**

In [ ]:
articles = ['2','3','5','6','8','9','10','11','14','P1-1']
n = len(articles)
matrix = np.zeros((n, n))

for (a, b), count in pairs.items():
    matrix[a][b] = count
    matrix[b][a] = count

plt.figure(figsize=(10, 8))
sns.heatmap(
    matrix,
    xticklabels=articles,
    yticklabels=articles,
    annot=True,
    fmt='.0f',
    cmap='Blues',
    linewidths=0.5
)
plt.title('Co-occurrence of ECHR Articles')
plt.tight_layout()
plt.show()

**Outliers**

In [ ]:
lengths = [len(case) for case in ds['train']['text']]
long = sum(1 for l in lengths if l > 50)
print(f"=== LONG CASES (>50 paragraphs) ===")
print(f"{long} cases ({long/len(lengths)*100:.1f}%) require truncation")
short = sum(1 for l in lengths if l < 3)
print(f"\n=== VERY SHORT CASES (<3 paragraphs) ===")
print(f"{short} cases - potentially problematic")

- 791 cases (8.8%) exceed 50 paragraphs and get truncated; only 3 are extremely short. Truncation is a real but limited concern (~9% of cases)

**Outliers Visualization**

In [ ]:
plt.figure(figsize=(10, 4))
plt.boxplot(lengths, vert=False)
plt.xlabel('Number of paragraphs')
plt.title('Outliers — Case Length (Boxplot)')
plt.tight_layout()
plt.show()

**Truncation Bias Check**

In [ ]:
df = pd.DataFrame({'length': lengths, 'labels': ds['train']['labels']})
df['truncated'] = df['length'] > 50
for i, art in enumerate(articles):
    rate_trunc = df[df['truncated']]['labels'].apply(lambda l: i in l).mean()
    rate_all = df['labels'].apply(lambda l: i in l).mean()
    print(f"{art}: {rate_trunc:.2f} (truncated) vs {rate_all:.2f} (overall)")


- Truncated cases skew heavily toward Article 2, 3, and 5 (life, torture, liberty) — 2.5–5x overrepresented vs their overall rate — while Article 6 and P1-1 are underrepresented among truncated cases. 
- Truncation isn't neutral — it disproportionately cuts evidence from the most severe rights violations (life, torture, detention) - so the optimal strategy is to summarize long cases rather than hard-truncate them.

**Train-Test Distribution Consistency Check + Validation**

In [ ]:
test_lengths = [len(case) for case in ds['test']['text']]
test_labels = [l for case in ds['test']['labels'] for l in case]
test_counter = Counter(test_labels)

val_lengths = [len(case) for case in ds['validation']['text']]
val_labels = [l for case in ds['validation']['labels'] for l in case]
val_counter = Counter(val_labels)

print("=== LENGTH: TRAIN vs VALIDATION vs TEST ===")
print(f"Train mean: {np.mean(lengths):.1f} paragraphs")
print(f"Validation mean: {np.mean(val_lengths):.1f} paragraphs")
print(f"Test mean: {np.mean(test_lengths):.1f} paragraphs")

print("\n=== ARTICLE DISTRIBUTION: TRAIN vs VALIDATION vs TEST ===")
for idx, art in enumerate(articles):
    print(f"Article {art}: train={counter[idx]}, val={val_counter[idx]}, test={test_counter[idx]}")

- Length is consistent across all three splits, and the Article 6 drop matches between validation and test — but Article 8 and Article 10 are notably more common in test than in either train or validation.
- Validation should be used for tuning, but shouldn't be fully trusted as a preview of test performance — Judge accuracy on Article 8/10 specifically should be checked once run on test, since validation doesn't warn about that shift.

**Train-Test Duplicate Check**

In [ ]:
# Exact duplicates

train_texts = set(" ".join(c) for c in ds['train']['text'])
test_texts = [" ".join(c) for c in ds['test']['text']]

exact_dupes = sum(1 for t in test_texts if t in train_texts)
print(f"Exact duplicates: {exact_dupes} / {len(test_texts)}")

In [ ]:
# Near duplicates - checking whether test-ds it shares at least a few exact paragraphs with train-ds

near_dupes = 0
for test_case in ds['test']['text'][:200]:
    test_paragraphs = set(test_case)
    for train_case in ds['train']['text']:
        shared = len(test_paragraphs & set(train_case))
        if shared >= 3:
            near_dupes += 1
            break

print(f"Near duplicates (200-sample from test): {near_dupes} / 200")

- No duplicates found in both duplicate checks